# Ch16: Resource Optimization（资源优化）— LangChain + MiMo

本章演示如何通过「模型路由」优化资源使用：
- 根据查询复杂度选择不同成本的模型
- 简单问题用便宜模型，复杂问题用贵模型
- 在质量和成本之间取得平衡

## 核心思想
不是所有问题都需要最强的模型。通过分类器将查询路由到合适的模型，
可以在保持质量的同时大幅降低成本。

## 第1步：导入依赖

In [1]:
import os
import json
import time

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

os.environ["OPENAI_API_KEY"] = os.getenv("MIMO_API_KEY", "")
os.environ["OPENAI_API_BASE"] = "https://token-plan-cn.xiaomimimo.com/v1"

# 模拟不同成本的模型（实际场景中可能是 gpt-4o-mini vs gpt-4o vs o1）
# 这里用不同 temperature 模拟「快速模型」和「深度推理模型」
fast_llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0.0,  # 低温 = 更快、更确定
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

reasoning_llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0.7,  # 高温 = 更多思考空间
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

print("模型初始化完成")
print("  - fast_llm: 快速模型（temperature=0.0）")
print("  - reasoning_llm: 推理模型（temperature=0.7）")

模型初始化完成
  - fast_llm: 快速模型（temperature=0.0）
  - reasoning_llm: 推理模型（temperature=0.7）


## 第2步：查询分类器

将查询分为两类：
- **simple**：简单事实性问题，直接回答
- **reasoning**：需要推理、分析的复杂问题

In [2]:
CLASSIFIER_PROMPT = """你是一个查询分类器。根据用户查询的复杂度，返回对应的类别。

只返回以下 JSON，不要有其他内容：
{ "classification": "simple" } 或 { "classification": "reasoning" }

规则：
- simple：直接事实性问题，不需要推理。例如：「法国首都是哪里？」「Python 的列表怎么排序？」
- reasoning：需要逻辑推理、多步分析、比较、评估。例如：「RAG 和微调哪个更好？」「分析这段代码的性能瓶颈」"""

def classify_query(query: str) -> str:
    """将查询分类为 simple 或 reasoning"""
    response = fast_llm.invoke([
        SystemMessage(content=CLASSIFIER_PROMPT),
        HumanMessage(content=query)
    ])
    text = response.content.strip()
    try:
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].split("```")[0].strip()
        result = json.loads(text)
        return result.get("classification", "simple")
    except json.JSONDecodeError:
        # 如果解析失败，根据关键词判断
        if "reasoning" in text.lower():
            return "reasoning"
        return "simple"

print("查询分类器定义完成")

查询分类器定义完成


## 第3步：路由器 + 执行

根据分类结果路由到不同的处理策略：
- **simple** → 快速回答，低 temperature
- **reasoning** → 深度推理，高 temperature + 思维链提示

In [3]:
REASONING_PROMPT = """你是一个深度推理助手。请仔细分析问题，分步骤思考，然后给出详细的回答。

请按以下格式回答：
1. **分析**：拆解问题的关键要素
2. **推理**：逐步分析每个要素
3. **结论**：给出最终答案"""

def route_and_execute(query: str) -> dict:
    """路由器：分类 → 选择模型 → 执行"""
    start_time = time.perf_counter()

    # 第1步：分类
    classification = classify_query(query)

    # 第2步：根据分类选择模型和 prompt
    if classification == "reasoning":
        llm = reasoning_llm
        system_prompt = REASONING_PROMPT
        model_name = "reasoning_llm (temp=0.7)"
    else:
        llm = fast_llm
        system_prompt = "你是一个简洁的助手，直接回答问题。"
        model_name = "fast_llm (temp=0.0)"

    # 第3步：执行
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ])

    elapsed = (time.perf_counter() - start_time) * 1000

    return {
        "query": query,
        "classification": classification,
        "model": model_name,
        "answer": response.content,
        "latency_ms": round(elapsed, 2)
    }

print("路由器定义完成")

路由器定义完成


## 第4步：测试路由器

用不同复杂度的查询测试路由效果。

In [4]:
# 测试用例

test_queries = [
    "法国的首都是哪里？",
    "Python 列表和元组有什么区别？",
    "比较 RAG 和微调的优缺点，什么场景下应该选择哪个？",
    "分析以下代码的时间复杂度并提出优化建议：def find_duplicates(arr): return [x for i, x in enumerate(arr) if arr.count(x) > 1]",
]

print("开始测试路由器...")
print("=" * 70)

results = []
for query in test_queries:
    result = route_and_execute(query)
    results.append(result)

    print(f"\n查询: {result['query']}")
    print(f"分类: {result['classification']}")
    print(f"模型: {result['model']}")
    print(f"耗时: {result['latency_ms']} ms")
    print(f"回答: {result['answer'][:100]}...")
    print("-" * 50)

开始测试路由器...



查询: 法国的首都是哪里？
分类: simple
模型: fast_llm (temp=0.0)
耗时: 8669.44 ms
回答: 法国的首都是**巴黎**。巴黎不仅是法国的政治、经济和文化中心，也是世界著名的艺术、时尚和旅游城市，常被称为“光之城”。...
--------------------------------------------------



查询: Python 列表和元组有什么区别？
分类: simple
模型: fast_llm (temp=0.0)
耗时: 15246.7 ms
回答: Python 列表和元组的主要区别如下：

1. **可变性**：列表是可变的，可以修改、添加或删除元素；元组是不可变的，创建后不能修改。
2. **语法**：列表使用方括号 `[]`，元组使用圆括号...
--------------------------------------------------



查询: 比较 RAG 和微调的优缺点，什么场景下应该选择哪个？
分类: reasoning
模型: reasoning_llm (temp=0.7)
耗时: 32044.64 ms
回答: ### 1. **分析**
问题要求比较 RAG（检索增强生成）和微调（Fine-tuning）的优缺点，并明确在什么场景下应该选择哪个。关键要素包括：
- **RAG 和微调的基本定义**：理解两者...
--------------------------------------------------



查询: 分析以下代码的时间复杂度并提出优化建议：def find_duplicates(arr): return [x for i, x in enumerate(arr) if arr.count(x) > 1]
分类: reasoning
模型: reasoning_llm (temp=0.7)
耗时: 27203.4 ms
回答: ### 分析
该函数的目的是找出列表 `arr` 中所有重复出现的元素。当前实现使用列表推导式，对列表中的每个元素调用 `arr.count(x)` 来判断其出现次数是否大于1。关键要素包括：
- 代...
--------------------------------------------------


## 第5步：成本分析

对比「全部用强模型」和「路由策略」的成本差异。

In [5]:
# 成本分析（模拟）

# 模拟不同模型的 token 成本（每 1000 tokens，单位：美元）
COST_FAST = 0.0001    # 便宜模型
COST_REASONING = 0.005 # 贵模型（50倍差价）

fast_count = sum(1 for r in results if r["classification"] == "simple")
reasoning_count = sum(1 for r in results if r["classification"] == "reasoning")

# 假设每次查询平均 500 input tokens + 300 output tokens = 800 tokens
avg_tokens = 800

cost_all_expensive = len(results) * avg_tokens / 1000 * COST_REASONING
cost_routed = (fast_count * avg_tokens / 1000 * COST_FAST) + (reasoning_count * avg_tokens / 1000 * COST_REASONING)

print("成本分析报告")
print("=" * 50)
print(f"总查询数: {len(results)}")
print(f"  简单查询: {fast_count} 次")
print(f"  推理查询: {reasoning_count} 次")
print(f"\n假设每次查询 {avg_tokens} tokens")
print(f"\n全部用贵模型: ${cost_all_expensive:.4f}")
print(f"路由策略:      ${cost_routed:.4f}")
print(f"节省:          ${cost_all_expensive - cost_routed:.4f} ({(1 - cost_routed/cost_all_expensive)*100:.1f}%)")

成本分析报告
总查询数: 4
  简单查询: 2 次
  推理查询: 2 次

假设每次查询 800 tokens

全部用贵模型: $0.0160
路由策略:      $0.0082
节省:          $0.0078 (49.0%)


## 总结

### 模型路由策略

```
用户查询 → 分类器 → simple → 快速模型（低成本）
                    → reasoning → 推理模型（高质量）
```

### 核心要点
- **不是所有问题都需要最强模型**——简单问题用便宜模型即可
- **分类器是关键**——分类准确率直接影响质量和成本
- **成本节省显著**——如果 70% 查询是简单的，可节省约 70% 成本

### 实际应用场景

| 场景 | 简单模型 | 复杂模型 |
|------|---------|----------|
| 客服系统 | FAQ 回答 | 投诉处理 |
| 代码助手 | 语法问题 | 架构设计 |
| 内容生成 | 标题/摘要 | 长文写作 |
| 数据分析 | 数据查询 | 趋势预测 |

### 进阶方向
- **多级路由**：不只是 2 级，可以是 3-4 级（mini → standard → pro → thinking）
- **动态路由**：根据实时延迟/错误率动态调整
- **Fallback 机制**：贵模型超时/报错时降级到便宜模型